# CNN-LSTM on Google Colab (GPU)

Trains the Order-Flow CNN-LSTM on a Colab **CUDA GPU**. This avoids the Apple-MPS
numerical bug entirely and is much faster, so we can use the dense paper settings.

## Before you run

1. **Enable the GPU**: `Runtime` -> `Change runtime type` -> Hardware accelerator = **GPU** (T4 is fine).
2. **Upload two things to your Google Drive** (anywhere; you'll set the path below):
   - `src/cnn_lstm.py`  (the training script)
   - `data/processed/of/`  (the 12 market `*.npz` files + `_config.json`, ~200 MB)

   The easiest way: drag the whole project folder `MS&E242_final/` into your Drive,
   keeping its `src/` and `data/processed/of/` subfolders.

> If you don't have the processed data yet, run `colab_of_pipeline.ipynb` first to
> generate `data/processed/of/`, then come back here.

## 1. Check the GPU

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Set Runtime -> Change runtime type -> GPU, then restart.')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Point to the project + verify the files

Edit `PROJECT_DIR` to wherever you put the project in Drive.

In [ ]:
from pathlib import Path

# >>> EDIT THIS to your project location in Drive <<<
PROJECT_DIR = Path('/content/drive/MyDrive/MS&E242_final')

SCRIPT   = PROJECT_DIR / 'src' / 'cnn_lstm.py'
DATA_DIR = PROJECT_DIR / 'data' / 'processed' / 'of'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SCRIPT.exists(), f'Missing script: {SCRIPT}'
assert (DATA_DIR / '_config.json').exists(), f'Missing data/_config.json in: {DATA_DIR}'
npz = sorted(DATA_DIR.glob('*.npz'))
npz = [p for p in npz if not p.name.startswith('_')]
print(f'Script : {SCRIPT}')
print(f'Data   : {DATA_DIR}  ({len(npz)} market npz files)')
print(f'Results: {RESULTS_DIR}')

## 4. Train on the GPU

Dense paper settings (`train_stride=10`, batch 512). On a T4 the full run
(~150k windows, up to 50 epochs with early stopping) takes only a few minutes.
All hyperparameters are CLI flags — tweak freely.

Each run appends a record to `results/experiments.jsonl` and writes a full
per-run JSON to `results/runs/cnn_lstm_<timestamp>.json`.

In [ ]:
!python "{SCRIPT}" \
    --data-dir "{DATA_DIR}" \
    --results-dir "{RESULTS_DIR}" \
    --device cuda \
    --train-stride 10 \
    --eval-stride 5 \
    --batch-size 512 \
    --lr 1e-3 \
    --max-epochs 50 \
    --patience 5 \
    --num-workers 2 \
    --tag colab-gpu

## 5. Read back the latest results

In [ ]:
import json

runs = sorted((RESULTS_DIR / 'runs').glob('cnn_lstm_*.json'))
rec = json.loads(runs[-1].read_text())
horizons = rec['horizons']
print('Run:', runs[-1].name, '| device:', rec['device'], '| params:', rec['n_params'])
print('Train windows:', rec['data']['n_windows'])

def show(title, m):
    print(f'\n{title}')
    hdr = '  ' + ' '.join(f'h={h:>4}' for h in horizons)
    print(hdr)
    for key in ['r2_oos', 'directional_accuracy', 'sharpe']:
        cells = ' '.join(f'{m[key][str(h)]:>6.3f}' for h in horizons)
        print(f'  {key:<22} {cells}')

show('OUT-OF-SAMPLE (test)', rec['metrics']['out_of_sample'])
show('IN-SAMPLE (train)',    rec['metrics']['in_sample'])
if rec['metrics'].get('linear_benchmark_oos'):
    show('LINEAR BENCHMARK (Ridge, OOS)', rec['metrics']['linear_benchmark_oos'])

## 6. (Optional) Run model variants for comparison

Re-run the cell below with different flags; every run is logged separately so you
can build the comparison table for the report.

In [ ]:
# Examples (uncomment one):
# Bigger LSTM:
# !python "{SCRIPT}" --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --device cuda --hidden 128 --batch-size 512 --tag hidden128
# Exact-spec architecture (no BatchNorm):
# !python "{SCRIPT}" --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --device cuda --no-batchnorm --lr 1e-4 --batch-size 512 --tag no-bn
# Lower learning rate:
# !python "{SCRIPT}" --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --device cuda --lr 5e-4 --batch-size 512 --tag lr5e-4